# Pista A — Motor empírico de simulación (ECO | Wind)

Sandbox de la **Fase 1 / Pista A** del plan técnico (`plan-tecnico-eco-wind.md`).
Objetivo: un pipeline trazable `simular(lat, lon, altura_buje, modelo, N) -> kWh_anual`
usando el catálogo de Flower Turbines ya validado (`engine/flower_turbines_curves.py`).

**Nota sobre entornos de ejecución:** este notebook se escribió y probó en un sandbox de
Claude Code sin salida de red a `power.larc.nasa.gov` (política de egress del entorno).
La celda de NASA POWER está armada para degradar sola a datos sintéticos cuando eso pasa,
así que el notebook corre de punta a punta en cualquier entorno — pero el resultado real
(con viento real del sitio) solo sale corriéndolo en **Google Colab** o cualquier entorno
con internet normal. Los pasos 3 y 4 (corrección de altura, ensamblado) sí están validados
acá con datos sintéticos.


In [1]:
import os

def _find_repo_root():
    # Busca un checkout existente del repo, mirando primero relativo al cwd
    # actual (dev sandbox, o Colab en una corrida anterior de esta misma
    # sesion) y despues la ruta estandar de Colab.
    for candidato in ("..", "/content/ECO-Wind"):
        if os.path.exists(os.path.join(candidato, ".git")):
            return os.path.abspath(candidato)
    return None

repo = _find_repo_root()
if repo is None:
    repo = "/content/ECO-Wind"
    get_ipython().system(f"git clone https://github.com/Sogo2012/ECO-Wind.git {repo}")
else:
    # Siempre forzar sync exacto con origin/main, sin importar el estado
    # previo del runtime (evita quedar pegado a una copia vieja).
    get_ipython().system(f"git -C {repo} fetch origin main")
    get_ipython().system(f"git -C {repo} reset --hard origin/main")

get_ipython().run_line_magic("cd", f"{repo}/notebooks")
get_ipython().system(f"git -C {repo} log -1 --format='Commit activo: %h  %s  (%ci)'")


Cloning into '/content/ECO-Wind'...


remote: Enumerating objects: 336, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (42/42), done.


remote: Total 336 (delta 41), reused 44 (delta 22), pack-reused 272 (from 1)
Receiving objects: 100% (336/336), 80.51 MiB | 17.37 MiB/s, done.


Resolving deltas: 100% (130/130), done.


/content/ECO-Wind/notebooks


Commit activo: cf317f7  feat(pista-b): Cilindro Actuador (efecto cluster) -- Paso 3 arranca  (2026-08-31 14:24:02 +0000)


## Paso 1 — Módulo base (`engine/flower_turbines_curves.py`)

In [2]:
import sys
sys.path.insert(0, "..")

import calendar
import numpy as np
import pandas as pd
import requests

from engine.flower_turbines_curves import (
    CURVE_COEFFICIENTS,
    power_isolated,
    bouquet_multiplier,
    power_in_bouquet,
)

print("Modelos disponibles:", list(CURVE_COEFFICIENTS))
print(f"Medium Tulip @ 12 m/s, aislada: {float(power_isolated(12, 'medium_tulip')):.1f} W")


Modelos disponibles: ['small_tulip', 'medium_tulip', 'three_m_tulip', 'large_tulip', 'al13_2m', 'al13_4m', 'al13_6m', 'al13_8m']
Medium Tulip @ 12 m/s, aislada: 622.2 W


## Paso 2 — Ingesta climática (NASA POWER Hourly)

Punto `community=SB`, parámetros `WS10M`/`WS50M`/`T2M`, formato JSON, año completo
(8,760 h ó 8,784 h en bisiesto).


In [3]:
NASA_POWER_HOURLY_URL = "https://power.larc.nasa.gov/api/temporal/hourly/point"


def fetch_nasa_power_hourly(lat, lon, year, community="SB",
                             parameters=("WS10M", "WS50M", "T2M")):
    """
    Descarga viento/temperatura horarios de NASA POWER para un año completo
    en una coordenada arbitraria. Devuelve un DataFrame con índice datetime
    horario y una columna por parámetro.

    NO EJECUTADO CON ÉXITO EN EL SANDBOX DE DESARROLLO: sin salida de red a
    power.larc.nasa.gov ahí. Validar en Colab (o cualquier entorno con
    internet normal) antes de confiar en el resultado.
    """
    params = {
        "parameters": ",".join(parameters),
        "community": community,
        "longitude": lon,
        "latitude": lat,
        "start": f"{year}0101",
        "end": f"{year}1231",
        "format": "JSON",
    }
    resp = requests.get(NASA_POWER_HOURLY_URL, params=params, timeout=60)
    resp.raise_for_status()
    param_data = resp.json()["properties"]["parameter"]
    df = pd.DataFrame(param_data)
    df.index = pd.to_datetime(df.index, format="%Y%m%d%H")
    df.index.name = "datetime"

    horas_esperadas = 8784 if calendar.isleap(year) else 8760
    if len(df) != horas_esperadas:
        raise ValueError(f"Esperaba {horas_esperadas} horas, llegaron {len(df)}")
    return df


In [4]:
def generar_clima_sintetico(year=2023, v_media=3.5, seed=42):
    """
    SOLO PARA PROBAR EL PIPELINE SIN RED. Serie horaria sintética con forma
    estacional simple (pico ilustrativo dic-abr, tipo estación seca de Costa
    Rica) + ruido Weibull. v_media=3.5 m/s es un valor ilustrativo, NO viene
    de ninguna fuente medida -- reemplazar por fetch_nasa_power_hourly() con
    dato real antes de sacar cualquier conclusión de negocio.
    """
    n_horas = 8784 if calendar.isleap(year) else 8760
    idx = pd.date_range(f"{year}-01-01", periods=n_horas, freq="h")
    rng = np.random.default_rng(seed)
    estacional = 1.0 + 0.4 * np.cos(2 * np.pi * (idx.dayofyear - 30) / 365)
    ruido = rng.weibull(2.0, size=n_horas)
    ws10m = np.clip(v_media * estacional * ruido / ruido.mean(), 0, None)
    return pd.DataFrame(
        {"WS10M": ws10m, "WS50M": ws10m * 1.15, "T2M": 22.0}, index=idx
    )


In [5]:
# Coordenada de prueba: Aeropuerto Juan Santamaría (misma zona que el EPW de
# datos_clima/, para comparar NASA POWER vs. estación manzanas con manzanas)
LAT, LON, YEAR = 10.00342327565566, -84.20332993360161, 2023

try:
    df_clima = fetch_nasa_power_hourly(LAT, LON, YEAR)
    print(f"OK -- {len(df_clima)} horas reales descargadas de NASA POWER.")
    datos_reales = True
except Exception as exc:
    print(f"No se pudo descargar de NASA POWER en este entorno ({exc!r}).")
    print("Sigo con datos SINTÉTICOS solo para probar el resto del pipeline.")
    df_clima = generar_clima_sintetico(YEAR, v_media=3.5)
    datos_reales = False

df_clima.head()


No se pudo descargar de NASA POWER en este entorno (ProxyError(MaxRetryError("HTTPSConnectionPool(host='power.larc.nasa.gov', port=443): Max retries exceeded with url: /api/temporal/hourly/point?parameters=WS10M%2CWS50M%2CT2M&community=SB&longitude=-84.20332993360161&latitude=10.00342327565566&start=20230101&end=20231231&format=JSON (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))).
Sigo con datos SINTÉTICOS solo para probar el resto del pipeline.


,WS10M,WS50M,T2M
2023-01-01 00:00:00,8.325665,9.574514,22.0
2023-01-01 01:00:00,8.207046,9.438103,22.0
2023-01-01 02:00:00,8.291923,9.535712,22.0
2023-01-01 03:00:00,2.840222,3.266255,22.0
2023-01-01 04:00:00,1.578642,1.815438,22.0


## Paso 2b — Fuente alternativa: EPW de estación real (offline)

Contraste con un archivo EPW/TMYx real (15 años de la estación del Aeropuerto Juan
Santamaría, `climate.onebuilding.org`, en `datos_clima/`). A diferencia de NASA POWER,
esto **no depende de la red** — funciona igual en este sandbox, en Colab, o donde sea —
y es un promedio de observaciones reales de 15 años en vez de una celda de reanálisis
satelital de ~50-60 km. La columna de viento del EPW ya está a 10m sobre el suelo
(misma referencia que WS10M), así que entra directo al mismo pipeline.

In [6]:
def load_epw_wind(path, year=2023):
    """
    Carga la velocidad de viento horaria de un archivo EPW (EnergyPlus
    Weather / TMYx, típicamente de climate.onebuilding.org).

    La columna 22 del EPW (índice 21) es Wind Speed a 10m sobre el suelo --
    misma altura de referencia que WS10M de NASA POWER, compatible directo
    con wind_at_height()/simular().

    year: año "etiqueta" para el índice datetime (un TMYx mezcla meses de
    años reales distintos -- no afecta promedios anuales/mensuales).
    """
    df = pd.read_csv(path, skiprows=8, header=None)
    idx = pd.date_range(f"{year}-01-01", periods=len(df), freq="h")
    return pd.DataFrame({"WS10M": df[21].values, "T2M": df[6].values}, index=idx)


EPW_PATH = "../datos_clima/CRI_AL_San.Jose-Santamaria.Intl.AP.787620_TMYx.2007-2021.epw"
df_epw = load_epw_wind(EPW_PATH)

fuente_actual = "NASA POWER real" if datos_reales else "NASA POWER (sintético, fallback -- sin red en este entorno)"
print(f"Media anual WS10M -- {fuente_actual}: {df_clima['WS10M'].mean():.2f} m/s")
print(f"Media anual WS10M -- EPW estación (15 años, aeropuerto): {df_epw['WS10M'].mean():.2f} m/s")
print(f"Horas con WS10M < cut-in (0.7 m/s) -- {fuente_actual}: {(df_clima['WS10M'] < 0.7).mean()*100:.1f}%")
print(f"Horas con WS10M < cut-in (0.7 m/s) -- EPW estación: {(df_epw['WS10M'] < 0.7).mean()*100:.1f}%")


Media anual WS10M -- NASA POWER (sintético, fallback -- sin red en este entorno): 3.51 m/s
Media anual WS10M -- EPW estación (15 años, aeropuerto): 4.03 m/s
Horas con WS10M < cut-in (0.7 m/s) -- NASA POWER (sintético, fallback -- sin red en este entorno): 4.4%
Horas con WS10M < cut-in (0.7 m/s) -- EPW estación: 3.3%


## Paso 2c — Fuente estadística: Global Wind Atlas (datos reales, confirmado a 10m)

Pablo exportó del panel del GWA para el punto del aeropuerto, confirmado a **10m** de altura: la curva de excedencia empírica de viento (50 puntos, `windSpeed.json`) y el mapa mes×hora real (`heatmapData.json`, 12×24). En vez de forzar un ajuste de Weibull de dos parámetros (que no calzaba bien -- la distribución anual mezcla dos regímenes estacionales muy distintos, seco y lluvioso), `generar_clima_gwa()` muestrea directamente de la curva empírica real (transformada inversa) y la escala por el índice real de cada mes×hora -- reproduce la magnitud Y la estacionalidad/ciclo diurno reales del sitio, sin asumir una forma paramétrica.

**Hallazgo aparte, con el `.lib` (formato WAsP nativo) que también compartió Pablo:** el archivo WAsP GWC da, para roughness class z0=0.030 (pasto corto, típico de aeropuerto) a 10m, una media ponderada por sector de **5.37 m/s** -- notablemente más alta que la del panel web del GWA (~3.67 m/s) y que el EPW real (4.03 m/s). Mi lectura: el `.lib` da el clima "generalizado" (reexpuesto a una rugosidad uniforme idealizada, pensado como insumo para un cálculo de micrositing completo en WAsP con el terreno real alrededor), mientras que el panel web ya hace ese downscaling específico para el punto exacto -- por eso uso el panel web (más cercano al EPW real) como fuente para `generar_clima_gwa()`, y dejo el `.lib` como referencia direccional (wind rose) más abajo, no como fuente de magnitud.

In [7]:
import calendar
import json


def cargar_gwa_json(carpeta):
    """Carga windSpeed.json (curva de excedencia) y heatmapData.json (indice mes x hora)
    de una carpeta con un export de plot data del Global Wind Atlas."""
    with open(f"{carpeta}/windSpeed.json") as f:
        ws = json.load(f)
    with open(f"{carpeta}/heatmapData.json") as f:
        hm = json.load(f)
    return ws, hm


def generar_clima_gwa(windspeed_json, heatmap_json, year=2023, seed=42):
    """
    Serie horaria de viento a partir del export real del Global Wind Atlas:
    - windspeed_json: lista de {perc, val} -- curva de excedencia empirica,
      perc = P(V > val)*100, en pasos de 2% de 2 a 100. Se usa transformada
      inversa (interpolacion) para muestrear de la distribucion REAL, sin
      asumir Weibull.
    - heatmap_json: lista de {month, hour, value} -- indice real mes x hora
      relativo a la media anual (promedio de los 288 valores ~= 1.0).

    Para cada hora del anio objetivo, sortea un valor normalizado de la
    curva de excedencia (media ~1) y lo escala por media_global *
    indice(mes, hora) de esa hora especifica -- reproduce la estacionalidad
    y el ciclo diurno reales sin duplicar la varianza (asume coeficiente de
    variacion constante a lo largo del anio -- aproximacion razonable, no
    exacta; sigue sin tener autocorrelacion dia a dia real).
    """
    perc = np.array([r["perc"] for r in windspeed_json], dtype=float)
    val = np.array([r["val"] for r in windspeed_json], dtype=float)
    orden = np.argsort(perc)
    perc_ord, val_ord = perc[orden], val[orden]
    media_global = val_ord.mean()

    idx_lookup = {(r["month"], r["hour"]): r["value"] for r in heatmap_json}

    n_horas = 8784 if calendar.isleap(year) else 8760
    idx_dt = pd.date_range(f"{year}-01-01", periods=n_horas, freq="h")
    rng = np.random.default_rng(seed)

    u = rng.uniform(0, 1, size=n_horas)
    perc_objetivo = 100 * (1 - u)  # perc = P(V>v)*100 -> u = P(V<=v)
    r_normalizado = np.interp(perc_objetivo, perc_ord, val_ord) / media_global

    factores = np.array([idx_lookup[(m, h)] for m, h in zip(idx_dt.month, idx_dt.hour)])
    ws = r_normalizado * media_global * factores

    return pd.DataFrame({"WS10M": ws, "T2M": 22.0}, index=idx_dt), media_global


GWA_DIR = "../datos_clima/gwa_juan_santamaria"
ws_json, hm_json = cargar_gwa_json(GWA_DIR)
df_gwa, media_global_gwa = generar_clima_gwa(ws_json, hm_json)

print(f"Media global GWA (confirmado 10m, panel web): {media_global_gwa:.3f} m/s")
print(f"Media anual de la serie generada: {df_gwa['WS10M'].mean():.3f} m/s (debe ~= la de arriba)")
print()
s = df_gwa["WS10M"]
print("Media por mes (debe parecerse a la estacionalidad real -- seco dic-abr, lluvioso may-oct):")
print(s.groupby(s.index.month).mean().round(2))


Media global GWA (confirmado 10m, panel web): 3.669 m/s
Media anual de la serie generada: 3.689 m/s (debe ~= la de arriba)

Media por mes (debe parecerse a la estacionalidad real -- seco dic-abr, lluvioso may-oct):
1     6.60
2     6.03
3     5.18
4     3.66
5     2.24
6     2.15
7     3.12
8     2.20
9     1.72
10    1.85
11    3.68
12    5.94
Name: WS10M, dtype: float64


## Paso 3 — Corrección de altura (perfil logarítmico)

In [8]:
def wind_at_height(v_ref, h_ref, h_target, z0=0.3):
    """
    Perfil logarítmico de viento: v(h) = v_ref * ln(h_target/z0) / ln(h_ref/z0)

    h_ref   : altura del dato de referencia (10 para WS10M, 50 para WS50M)
    h_target: altura real de buje de la turbina
    z0      : longitud de rugosidad (m). Default 0.3 = suburbano/urbano bajo.
              Ajustar por sitio: 0.03 campo abierto, 0.1 cultivos bajos,
              0.3 suburbano (default), 1.0 urbano denso.

    Las turbinas Flower Turbines son muy bajas (buje entre 1 y 6 m), casi
    siempre POR DEBAJO de los 10 m de referencia -- a diferencia de una HAWT
    grande (buje 80m+), acá la corrección casi siempre REDUCE la velocidad
    respecto al dato crudo de NASA POWER.
    """
    v_ref = np.asarray(v_ref, dtype=float)
    return v_ref * np.log(h_target / z0) / np.log(h_ref / z0)


# Autotest con valores sintéticos (no depende de la red)
print("Autotest perfil logarítmico, v_10m = 5.00 m/s:")
for h in [1.4, 3.0, 6.0, 10.0]:
    print(f"  altura buje={h:4.1f} m  ->  v={wind_at_height(5.0, 10, h):.2f} m/s")
assert abs(wind_at_height(5.0, 10, 10) - 5.0) < 1e-9, "en h_target=h_ref debe dar v_ref exacto"
print("OK: en h_target = h_ref da v_ref exacto.")


Autotest perfil logarítmico, v_10m = 5.00 m/s:
  altura buje= 1.4 m  ->  v=2.20 m/s
  altura buje= 3.0 m  ->  v=3.28 m/s
  altura buje= 6.0 m  ->  v=4.27 m/s
  altura buje=10.0 m  ->  v=5.00 m/s
OK: en h_target = h_ref da v_ref exacto.


## Paso 4 — Ensamblar `simular(lat, lon, altura_buje, modelo, N) -> kWh_anual`

In [9]:
def simular(df_clima, altura_buje, modelo, N, h_ref=10, z0=0.3, metodo_bouquet="real"):
    """
    Ensambla la serie horaria de potencia del clúster y la agrega a kWh
    mensual/anual, usando P(v) = k*v^3 x M(N) del motor empírico.

    df_clima: DataFrame con índice datetime horario y columna 'WS10M' (m/s)
              -- viene de fetch_nasa_power_hourly() o de datos sintéticos;
              la función no sabe ni le importa el origen.
    """
    v_hub = wind_at_height(df_clima["WS10M"].values, h_ref, altura_buje, z0=z0)
    potencia_w_por_turbina = power_in_bouquet(v_hub, modelo, N, metodo=metodo_bouquet)

    serie = pd.Series(potencia_w_por_turbina, index=df_clima.index,
                       name="potencia_W_por_turbina")
    energia_cluster_kwh = serie * N / 1000.0
    return {
        "serie_horaria_W_por_turbina": serie,
        "kwh_mensual": energia_cluster_kwh.resample("MS").sum(),
        "kwh_anual": float(energia_cluster_kwh.sum()),
    }


resultado = simular(df_clima, altura_buje=3.0, modelo="medium_tulip", N=3)
etiqueta = "REAL (NASA POWER)" if datos_reales else "SINTÉTICO -- no es dato real del sitio"
print(f"Escenario: Medium Tulip x3 (bouquet), buje a 3.0 m, San José CR, {YEAR} [{etiqueta}]")
print(f"kWh/año: {resultado['kwh_anual']:.1f}")
print()
print(resultado["kwh_mensual"])


Escenario: Medium Tulip x3 (bouquet), buje a 3.0 m, San José CR, 2023 [SINTÉTICO -- no es dato real del sitio]
kWh/año: 426.6

2023-01-01    78.248672
2023-02-01    69.182152
2023-03-01    64.546197
2023-04-01    38.412973
2023-05-01    21.077526
2023-06-01     9.701907
2023-07-01     6.164773
2023-08-01     6.898122
2023-09-01    10.271701
2023-10-01    20.338143
2023-11-01    36.439241
2023-12-01    65.307598
Freq: MS, Name: potencia_W_por_turbina, dtype: float64


In [10]:
print(f"{'Buje (m)':>10}  {'v_hub medio (m/s)':>18}  {'% horas<cutin':>14}  {'kWh/año':>10}")
for altura in [1.4, 3.0, 6.0]:
    r_epw = simular(df_epw, altura_buje=altura, modelo="medium_tulip", N=3)
    v_hub_epw = wind_at_height(df_epw["WS10M"].values, 10, altura, z0=0.3)
    pct = (v_hub_epw < 0.7).mean() * 100
    print(f"{altura:>10.1f}  {v_hub_epw.mean():>18.2f}  {pct:>13.1f}%  {r_epw['kwh_anual']:>10.1f}")

print()
print("(Mismo escenario que arriba: Medium Tulip x3 en bouquet, z0=0.3.)")
print(f"Para comparar: con [{etiqueta}] a 3.0m dio {resultado['kwh_anual']:.1f} kWh/año.")


  Buje (m)   v_hub medio (m/s)   % horas<cutin     kWh/año
       1.4                1.77           19.8%       181.3
       3.0                2.65           10.2%       606.8
       6.0                3.45            3.4%      1337.0

(Mismo escenario que arriba: Medium Tulip x3 en bouquet, z0=0.3.)
Para comparar: con [SINTÉTICO -- no es dato real del sitio] a 3.0m dio 426.6 kWh/año.


In [11]:
r_gwa = simular(df_gwa, altura_buje=3.0, modelo="medium_tulip", N=3)
print(f"kWh/año [GWA real, panel web, buje 3.0m]: {r_gwa['kwh_anual']:.1f}")
print(f"(Para comparar: EPW real dio 606.8 kWh/año, NASA POWER real dio 16.5 kWh/año, "
      f"mismo escenario y misma coordenada.)")

# --- Bonus: wind rose direccional real, desde el .lib de WAsP (no se usa para la magnitud,
# solo como referencia de direccion dominante -- ver nota del Paso 2c sobre la discrepancia
# de magnitud entre el .lib y el panel web) ---
with open(f"{GWA_DIR}/gwc_point_1_10m.lib") as f:
    lib_lines = [l.split() for l in f.readlines()]
z0_vals = [float(x) for x in lib_lines[2]]
heights = [float(x) for x in lib_lines[3]]
idx_lib = 4
bloques_lib = {}
for z0 in z0_vals:
    freq = [float(x) for x in lib_lines[idx_lib]]; idx_lib += 1
    for h in heights:
        A = [float(x) for x in lib_lines[idx_lib]]; idx_lib += 1
        k = [float(x) for x in lib_lines[idx_lib]]; idx_lib += 1
        bloques_lib[(z0, h)] = {"freq": freq, "A": A, "k": k}

freq_10m = bloques_lib[(0.030, 10.0)]["freq"]
print()
print("Wind rose direccional (.lib, z0=0.030, 10m) -- sectores dominantes:")
sectores = sorted(range(12), key=lambda i: -freq_10m[i])[:3]
for i in sectores:
    grados = i * 30
    print(f"  Sector {i+1} ({grados}°-{grados+30}°): {freq_10m[i]:.1f}% del tiempo")


kWh/año [GWA real, panel web, buje 3.0m]: 387.9
(Para comparar: EPW real dio 606.8 kWh/año, NASA POWER real dio 16.5 kWh/año, mismo escenario y misma coordenada.)

Wind rose direccional (.lib, z0=0.030, 10m) -- sectores dominantes:
  Sector 4 (90°-120°): 25.0% del tiempo
  Sector 5 (120°-150°): 24.5% del tiempo
  Sector 9 (240°-270°): 15.4% del tiempo


## Paso 5 — Sanity check de orden de magnitud

In [12]:
# Referencia externa (Kilowatts UK, distribuidor UK): 1,000-5,000 kWh/año
# típico para turbinas pequeñas. No es para copiar el número -- Costa Rica
# tiene otro recurso eólico y contexto -- solo para confirmar que el orden
# de magnitud del resultado no es absurdo.
REF_UK_BAJO, REF_UK_ALTO = 1000, 5000
kwh = resultado["kwh_anual"]

print(f"Resultado [{etiqueta}]: {kwh:.0f} kWh/año")
print(f"Referencia UK (orden de magnitud, turbinas pequeñas): {REF_UK_BAJO}-{REF_UK_ALTO} kWh/año")

if REF_UK_BAJO * 0.3 <= kwh <= REF_UK_ALTO * 3:
    print("-> Orden de magnitud razonable.")
else:
    print("-> FUERA de rango incluso con margen amplio -- revisar el pipeline antes de confiar en él.")

if not datos_reales:
    print()
    print("!! Este sanity check corrió sobre datos SINTÉTICOS. Repetir en Colab con NASA POWER")
    print("   real antes de sacar cualquier conclusión sobre el recurso eólico de Costa Rica.")


Resultado [SINTÉTICO -- no es dato real del sitio]: 427 kWh/año
Referencia UK (orden de magnitud, turbinas pequeñas): 1000-5000 kWh/año
-> Orden de magnitud razonable.

!! Este sanity check corrió sobre datos SINTÉTICOS. Repetir en Colab con NASA POWER
   real antes de sacar cualquier conclusión sobre el recurso eólico de Costa Rica.


## Resumen y próximos pasos

- **Comparación final de fuentes climáticas, misma coordenada exacta (Aeropuerto Juan Santamaría), mismo escenario (Medium Tulip x3, buje 3.0m):**

| Fuente | Viento medio 10m | kWh/año |
|---|---|---|
| NASA POWER (real) | 1.30 m/s | 16.5 |
| Global Wind Atlas (real, panel web, 10m confirmado) | 3.67 m/s | 387.9 |
| EPW estación real (15 años, TMYx) | 4.03 m/s | 606.8 |

- **NASA POWER queda descartado como fuente primaria** -- confirmado con evidencia (Hallazgo previo).
- **GWA y EPW concuerdan razonablemente entre sí** (~9% de diferencia en la media de viento) y los dos están en el mismo orden de magnitud de energía -- ambos son fuentes utilizables. El EPW es más alto porque son observaciones reales de estación; el GWA es un modelo (downscaling a 250m), buen respaldo para sitios sin estación cercana.
- **Hallazgo metodológico sobre el GWA:** el archivo `.lib` (formato WAsP nativo, con el detalle por rugosidad/altura/sector) da una media notablemente más alta (5.37 m/s a z0=0.030, 10m) que el panel web (3.67 m/s). Interpretación de trabajo: el `.lib` es el clima "generalizado" re-expuesto a una rugosidad idealizada uniforme (insumo para un cálculo WAsP completo con el terreno real), mientras que el panel web ya hace el downscaling específico para el punto exacto -- por eso se usa el panel web para la magnitud, y el `.lib` solo para el wind rose direccional (sectores dominantes: 90°-150°, ~50% del tiempo combinado).
- **Validado en este sandbox, de punta a punta y con datos reales:** `engine/flower_turbines_curves.py` (Paso 1), corrección de altura (Paso 3, autotest), `simular()` (Paso 4), contra EPW (Paso 2b) y contra GWA real confirmado a 10m (Paso 2c) -- ambos offline, sin depender de la red.
- **Pendiente:** confirmar si la brecha GWA-vs-EPW (~9% en viento) se sostiene en otros sitios: si sí, podría valer la pena un factor de corrección simple; conseguir EPWs/exports de GWA de los sitios reales de proyectos; decidir si vale la pena la Pista C (ERA5 + bias correction) dado que GWA y EPW ya dan resultados razonablemente consistentes entre sí.
- **Variables abiertas** (sección 4 del plan, no bloqueantes): calibración K(v) contra datos de campo reales, z0 real del sitio (se usó 0.3 como default ilustrativo en `wind_at_height`, distinto del z0=0.030 usado para leer el `.lib` -- vale la pena unificar), validez del M(N) exponencial más allá de N=10 o en layouts 2D.
